# Day 10 · Delta Lake Fundamentals
## Storage that remembers

**Databricks + Snowflake · 70-Hour Programme · DataTrends.tech**

---

| Day | What you gained |
|---|---|
| 7 · 8 | Transformations, actions, and `explain()` |
| 9 | SQL over DataFrames — views, the catalog, UDFs |
| **10** | **Tables that survive the session, and remember every change** |

Every name created last night is already gone. The source table is not.
Tonight is the difference between those two sentences.

## Configuration

In [ ]:
from pyspark.sql import functions as F

CATALOG      = "workspace"
SCHEMA       = "day10"
SOURCE_TABLE = "workspace.default.upi_transactions_2026"

T = f"{CATALOG}.{SCHEMA}.upi_delta"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"DROP TABLE IF EXISTS {T}")

print(f"source : {SOURCE_TABLE}")
print(f"target : {T}")

---
# 1 · What a table actually is

**Industry example.** A payments team runs a nightly load. At 02:00 the job crashes
halfway. In the morning the analysts still query the table. Somebody has to be able to
say — with certainty — whether they are looking at last night's data or half of it.

Before we build anything, look at what is already here.

In [ ]:
display(spark.sql(f"DESCRIBE DETAIL {SOURCE_TABLE}"))

### It was Delta the whole time

Read the `format` column. Every managed table in Unity Catalog is created as Delta unless
somebody goes out of their way to ask for something else — so the table you have been
querying since Day 6 has been a Delta table all along.

You saw the evidence yesterday without knowing it. In the physical plan, the scan read
`PreparedDeltaFileIndex` — not a plain list of Parquet files, but a file index built from a
transaction log.

So a Delta table is two things: **Parquet files**, and a **transaction log** that says which
of those files count right now. Tonight is about the second one — and we build our own so
that we can break it safely.

In [ ]:
# One city held back, as though its file had not arrived yet.
initial = (
    spark.table(SOURCE_TABLE)
         .filter(F.col("city") != "Chennai")
)

initial.write.format("delta").mode("overwrite").saveAsTable(T)

display(spark.sql(f"SELECT COUNT(*) AS rows FROM {T}"))

### What was created

In [ ]:
display(spark.sql(f"DESCRIBE DETAIL {T}"))

### The log

`DESCRIBE HISTORY` is the log, read as a table. One row per commit, for the life of the
table — who changed it, when, with what operation, and how much it touched.

This is the thing a Parquet directory does not have, and it is the whole difference.

In [ ]:
display(spark.sql(f"DESCRIBE HISTORY {T}"))

---
# 2 · ACID — the promise the log makes

| | What it means here |
|---|---|
| **A**tomic | a write either commits completely or not at all |
| **C**onsistent | the table never contradicts its own schema |
| **I**solated | readers see one committed version, never a half-written one |
| **D**urable | once committed, it survives everything |

The next file arrives.

In [ ]:
todays_file = (
    spark.table(SOURCE_TABLE)
         .filter(F.col("city") == "Chennai")
)

todays_file.write.format("delta").mode("append").saveAsTable(T)

display(spark.sql(f"SELECT COUNT(*) AS rows FROM {T}"))

In [ ]:
display(spark.sql(f"DESCRIBE HISTORY {T}"))

### Atomicity, demonstrated

A write that does not satisfy the table's contract is rejected. The interesting question
is not whether it fails — it is what the table looks like afterwards.

In [ ]:
before_rows    = spark.sql(f"SELECT COUNT(*) AS n FROM {T}").collect()[0]["n"]
before_commits = spark.sql(f"DESCRIBE HISTORY {T}").count()

bad_batch = todays_file.withColumn("settlement_ref", F.lit("SETL-0001"))

try:
    bad_batch.write.format("delta").mode("append").saveAsTable(T)
    print("the write was accepted")
except Exception as e:
    print(type(e).__name__)
    print(str(e).split("\n")[0][:300])

In [ ]:
after_rows    = spark.sql(f"SELECT COUNT(*) AS n FROM {T}").collect()[0]["n"]
after_commits = spark.sql(f"DESCRIBE HISTORY {T}").count()

print(f"rows    before: {before_rows:>12,}   after: {after_rows:>12,}")
print(f"commits before: {before_commits:>12}   after: {after_commits:>12}")

---
# 3 · Time travel

**Industry example.** A cleanup script written for the staging environment is run against
production at 21:40 on a Friday. It removes a city. The rows are gone and the job reported
success.

On a plain data lake, that is a restore-from-backup conversation and a long weekend.

In [ ]:
spark.sql(f"DELETE FROM {T} WHERE city = 'Hyderabad'")

display(spark.sql(f"SELECT COUNT(*) AS rows FROM {T}"))

In [ ]:
display(spark.sql(f"DESCRIBE HISTORY {T}"))

### Reading the past

The old files were never deleted. The log simply stopped counting them.

In [ ]:
display(spark.sql(f"""
    SELECT 0 AS version, COUNT(*) AS rows FROM {T} VERSION AS OF 0
    UNION ALL
    SELECT 1, COUNT(*) FROM {T} VERSION AS OF 1
    UNION ALL
    SELECT 2, COUNT(*) FROM {T} VERSION AS OF 2
    ORDER BY version
"""))

### By clock time, not version number

`TIMESTAMP AS OF` resolves to the latest version committed **at or before** that instant.
In production you rarely know a version number — you know when the report was right.

In [ ]:
ts_v1 = (spark.sql(f"DESCRIBE HISTORY {T}")
              .filter("version = 1")
              .select("timestamp")
              .collect()[0]["timestamp"])

print(f"version 1 was committed at {ts_v1}")

display(spark.sql(f"SELECT COUNT(*) AS rows FROM {T} TIMESTAMP AS OF '{ts_v1}'"))

### Putting it back

In [ ]:
display(spark.sql(f"RESTORE TABLE {T} TO VERSION AS OF 1"))

In [ ]:
display(spark.sql(f"SELECT COUNT(*) AS rows FROM {T}"))

In [ ]:
display(spark.sql(f"DESCRIBE HISTORY {T}"))

### The part worth noticing

The restore did not erase the mistake. It appended a new commit that reinstates the old
set of files. The deletion is still in the history, and so is the recovery.

Which means the undo can itself be undone. Nothing in this table is ever quietly rewritten.

---
# 4 · Schema enforcement, and schema evolution

Section 2 showed the table refusing a column it had not agreed to. That refusal is
**schema enforcement**, and it is on by default — the table protects its own contract.

Sometimes the new column is legitimate and the contract should change. That has to be
something you say out loud, in the code, on purpose.

In [ ]:
(bad_batch.write
   .format("delta")
   .mode("append")
   .option("mergeSchema", "true")
   .saveAsTable(T))

spark.sql(f"DESCRIBE {T}").show(truncate=False)

### What happened to the rows that came before

They were not rewritten. They simply have no value for a column that did not exist when
they were written.

In [ ]:
display(spark.sql(f"""
    SELECT CASE WHEN settlement_ref IS NULL THEN 'written before the column existed'
                ELSE 'written after'
           END AS origin,
           COUNT(*) AS rows
    FROM {T}
    GROUP BY 1
"""))

### The rule to leave with

Enforcement is the default because silent schema drift is how a warehouse rots. Evolution
is available because requirements genuinely change. `mergeSchema` is you signing for it.

---
# 5 · One format, two readers

Delta and Apache Iceberg solve the same problem and are not compatible with each other.
For years that forced a choice: pick the format your engine reads, and accept that the
other half of the organisation cannot read it.

**Delta UniForm** writes the Iceberg metadata alongside the Delta metadata, over one copy
of the data. An Iceberg reader can then read a Delta table without a conversion job.

This matters for us specifically: it is the mechanism by which the Databricks half of this
course and the Snowflake half stop being separate worlds.

In [ ]:
try:
    spark.sql(f"""
        ALTER TABLE {T} SET TBLPROPERTIES (
            'delta.columnMapping.mode'                 = 'name',
            'delta.enableIcebergCompatV2'              = 'true',
            'delta.universalFormat.enabledFormats'     = 'iceberg'
        )
    """)
    print("UniForm enabled on this table")
except Exception as e:
    print("not enabled on this workspace")
    print(type(e).__name__)
    print(str(e).split("\n")[0][:300])

In [ ]:
display(spark.sql(f"SHOW TBLPROPERTIES {T}"))

---
## Your turn — 28 minutes

Work in your own workspace on `samples.nyctaxi.trips`. Nothing to download.

```
trips : tpep_pickup_datetime · tpep_dropoff_datetime · trip_distance
        fare_amount · pickup_zip · dropoff_zip
```

**1 — Make a table.**
Create a schema of your own. Write the trips where `fare_amount` is under 20 into a Delta
table called `trips_delta`. Read the row count back.

**2 — Read the log.**
Run `DESCRIBE DETAIL` and `DESCRIBE HISTORY`. How many files, how many commits, what
operation is recorded?

**3 — Add, then break, then look.**
Append the trips where `fare_amount` is between 20 and 50. Then delete every trip with a
`trip_distance` of 0. Check the history after each step.

**4 — Go back.**
Report the row count at every version using `VERSION AS OF`. Then restore the table to the
version just before your delete, and show that the restore is itself a new version.

**5 — Refuse, then allow.**
Add a column `audit_flag` to a batch and try to append it. Read the error. Then append it
properly with `mergeSchema`, and count how many rows have a null in the new column.

**6 — Stretch.**
Two versions of your table differ by one delete. Without using `DESCRIBE HISTORY`, write a
query that returns the rows that were removed.

---
# What tonight did not give you

You can add rows and remove rows. You cannot yet **update** a row that already exists, or
apply a file of changes where some rows are new and some are edits — which is what every
real source system sends you.

You also now have a table that keeps every version forever, and nobody has mentioned what
that costs.

Both are tomorrow.

## Reset

Removes everything this notebook created. The source table is untouched.

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {T}")
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.{SCHEMA} CASCADE")

print("session objects removed")